In [1]:
import pandas as pd
from sklearn.cluster import KMeans
df = pd.read_csv("../data/cleaned_data.csv")
buy_df = df[df['behavior']=='buy']

In [2]:
# 1. 计算RFM
# R：最后一次购买距离2017-12-03的天数
# F：购买次数
# M：（数据集无金额，不影响项目质量）
current_date = pd.to_datetime('2017-12-04')
rfm = buy_df.groupby('user_id').agg(
    R=('time', lambda x: (current_date - pd.to_datetime(x).max()).days),
    F=('behavior','count')
).reset_index()

# 2. K-Means聚类（分成3类用户）
kmeans = KMeans(n_clusters=3, n_init=10,init='k-means++',random_state=42)
rfm['user_type'] = kmeans.fit_predict(rfm[['R','F']])

# 3. 业务命名
rfm['user_type_name'] = rfm['user_type'].map({
    0:'流失用户',
    1:'高价值用户',
    2:'潜力用户'
})

# 保存结果
rfm.to_csv("../result/rfm_user.csv",index=False)
print("✅ RFM+K-Means用户分层完成！")
print(rfm['user_type_name'].value_counts())

✅ RFM+K-Means用户分层完成！
user_type_name
潜力用户     381181
流失用户     210563
高价值用户     78626
Name: count, dtype: int64
